[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Y-Robin/DeepLearningKurs/blob/main/notebooks/Day_3_5/15_tag_3_5_custom_training_loop_gradient_tape.ipynb)

# Tag 3-5 - 15 Custom Training Loop mit GradientTape

`model.fit(...)` ist für die meisten Projekte der richtige Einstieg. Trotzdem ist es wichtig, einmal zu sehen, was darunter passiert:

1. Batch laden,
2. Vorhersage berechnen,
3. Loss berechnen,
4. Gradienten berechnen,
5. Gewichte aktualisieren,
6. Metriken aktualisieren.

Dafür verwenden wir `tf.GradientTape`. Das ist der Baustein, mit dem TensorFlow automatische Ableitungen für eigene Trainingsschleifen bereitstellt.


## Lesepfad für dieses Notebook

Die wichtigsten Objekte und wo sie wieder auftauchen:

- `train_ds`: enthält Trainingsbatches. Wird in der Trainingsschleife mit `for x_batch, y_batch in train_ds:` durchlaufen.
- `val_ds`: enthält Validation-Batches. Wird nur zur Bewertung benutzt, nicht zum Gewichte-Update.
- `model`: neuronales Netz.
- `loss_fn`: berechnet den Fehler.
- `optimizer`: entscheidet, wie Gewichte mit den Gradienten aktualisiert werden.
- `train_step(...)`: verarbeitet genau einen Trainingsbatch.
- `val_step(...)`: verarbeitet genau einen Validation-Batch.

`next(iter(train_ds))` bedeutet immer nur: “Gib mir einmal den ersten Batch, damit ich sehen kann, wie die Daten aussehen.” Es trainiert noch nichts dauerhaft.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
tf.keras.utils.set_random_seed(RANDOM_STATE)
plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True


## Kurz erklärt: Objekt, Methode, Iterator

Ein häufiger Stolperstein in diesem Notebook ist die Schreibweise mit Punkten:

```python
train_ds = tf.data.Dataset.from_tensor_slices(...)
train_ds = train_ds.shuffle(...).batch(...)
```

`train_ds` ist keine normale Funktion, sondern ein **Objekt** vom Typ `tf.data.Dataset`. Ein Objekt kann Methoden besitzen. Methoden sind Funktionen, die zu diesem Objekt gehören. Deshalb darf man schreiben:

- `train_ds.shuffle(...)`: Methode des Dataset-Objekts, mischt die Elemente.
- `train_ds.batch(32)`: Methode des Dataset-Objekts, bündelt einzelne Samples zu Batches.
- `train_ds.prefetch(...)`: Methode des Dataset-Objekts, bereitet Daten effizient vor.

Wichtig: Diese Methoden verändern das alte Dataset meist nicht direkt. Sie geben ein neues Dataset zurück. Deshalb kann man sie hintereinanderketten:

```python
train_ds = dataset.shuffle(...).batch(...).prefetch(...)
```

`iter(train_ds)` erzeugt einen Iterator. Ein Iterator ist etwas, das man Schritt für Schritt abfragen kann. `next(iter(train_ds))` holt genau das nächste Element, hier also einen Batch. Das ist nur ein Kontrollblick, kein komplettes Training.


## Daten und `tf.data`

Für eigene Trainingsschleifen ist `tf.data.Dataset` sehr praktisch. Es übernimmt Batching, Shuffling und Prefetching.


In [ ]:
# X/y sind noch normale NumPy-Arrays.
X, y = make_moons(n_samples=1800, noise=0.25, random_state=RANDOM_STATE)
X = StandardScaler().fit_transform(X).astype("float32")
# reshape(-1, 1) macht aus y die Form (samples, 1), passend zum Sigmoid-Output.
y = y.reshape(-1, 1).astype("float32")

# Split: train_ds wird zum Lernen benutzt, val_ds nur zur Kontrolle nach jeder Epoche.
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y.ravel()
)

batch_size = 32

# tf.data.Dataset.from_tensor_slices(...) ist eine TensorFlow-Funktion.
# Sie macht aus NumPy-Arrays ein Dataset-Objekt.
# Dieses Objekt hat danach Methoden wie .shuffle(...), .batch(...) und .prefetch(...).
train_ds = (
    tf.data.Dataset.from_tensor_slices((X_train, y_train))
    # shuffle mischt Samples vor dem Batching. buffer_size=len(X_train) heißt: vollständig mischen.
    .shuffle(buffer_size=len(X_train), seed=RANDOM_STATE)
    # batch(32): aus einzelnen Samples werden Mini-Batches mit 32 Samples.
    .batch(batch_size)
    # prefetch bereitet den nächsten Batch vor, während der aktuelle Batch trainiert wird.
    .prefetch(tf.data.AUTOTUNE)
)
# Bei val_ds nutzen wir kein shuffle, weil Validation reproduzierbar und stabil bleiben soll.
# .batch(...) ist aber auch hier sinnvoll, damit evaluate/val_step in Batches arbeiten kann.
val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val)).batch(batch_size)

# Nur zur Kontrolle: Einmal einen Batch aus train_ds herausnehmen und Form ansehen.
# next(iter(...)) startet keine Epoche, sondern holt nur ein Beispiel-Element aus dem Dataset.
example_batch = next(iter(train_ds))
example_batch[0].shape, example_batch[1].shape


## Modell, Loss, Optimizer und Metriken

Beim Custom Loop müssen wir die Bausteine explizit anlegen. Keras nimmt uns nicht mehr automatisch alles ab.

Die wichtigste Stelle ist der Optimizer:

- `optimizer = keras.optimizers.Adam(learning_rate=0.02)` bestimmt **welches Update-Verfahren** benutzt wird.
- `learning_rate=0.02` bestimmt die Schrittgröße.
- Wenn man statt Adam klassisches SGD mit Momentum sehen möchte, könnte man z. B. `keras.optimizers.SGD(learning_rate=0.02, momentum=0.9)` verwenden.
- Die Trainingsschleife unten bleibt gleich. Nur die Optimizer-Zeile ändert sich.

Der Loss bestimmt, was minimiert wird. Die Metriken protokollieren nur, wie gut das Modell wirkt; sie aktualisieren keine Gewichte.


In [ ]:
def build_model():
    return keras.Sequential(
        [
            keras.Input(shape=(2,), name="features"),
            layers.Dense(32, activation="relu"),
            layers.Dense(16, activation="relu"),
            layers.Dense(1, activation="sigmoid"),
        ],
        name="custom_loop_classifier",
    )


model = build_model()
loss_fn = keras.losses.BinaryCrossentropy()

# Hier wird der Optimizer und die Learning Rate festgelegt.
# Adam enthält intern adaptive Schrittgrößen und Momentum-ähnliche gleitende Mittel.
optimizer = keras.optimizers.Adam(learning_rate=0.02)

# Alternative zum Ausprobieren:
# optimizer = keras.optimizers.SGD(learning_rate=0.02, momentum=0.9)

train_loss = keras.metrics.Mean(name="train_loss")
train_acc = keras.metrics.BinaryAccuracy(name="train_accuracy")
val_loss = keras.metrics.Mean(name="val_loss")
val_acc = keras.metrics.BinaryAccuracy(name="val_accuracy")


## Zwischenstufe: eigener Loop ohne `GradientTape`

Bevor wir ganz tief mit `GradientTape` einsteigen, kann man einen einfacheren eigenen Loop bauen.

Hier machen wir die äußere Schleife selbst:

- Wir laufen über Epochen.
- Wir laufen über Batches.
- Wir protokollieren Ergebnisse selbst.

Aber: Die eigentliche Gewichtsaktualisierung macht Keras intern über `train_on_batch(...)`. Das ist also “custom”, aber noch nicht komplett von Hand.

`train_on_batch(x_batch, y_batch)` nutzt genau das, was vorher in `compile(...)` festgelegt wurde:

- Optimizer, z. B. Adam oder SGD,
- Learning Rate,
- Loss,
- Metriken.

Wir brauchen hier kein `GradientTape`, weil Keras die Gradienten intern berechnet.


In [ ]:
# Eigenes Modell nur für diese Zwischenstufe, damit der spätere GradientTape-Loop sauber separat bleibt.
simple_loop_model = build_model()
simple_loop_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.02),
    loss=keras.losses.BinaryCrossentropy(),
    metrics=[keras.metrics.BinaryAccuracy(name="accuracy")],
)

simple_history = []
simple_epochs = 8

for epoch in range(1, simple_epochs + 1):
    batch_results = []

    # train_ds liefert schon Batches, weil oben .batch(batch_size) aufgerufen wurde.
    for x_batch, y_batch in train_ds:
        # train_on_batch trainiert genau auf diesem einen Batch.
        # Intern passieren Vorhersage, Loss, Gradienten und Optimizer-Update.
        result = simple_loop_model.train_on_batch(x_batch, y_batch, return_dict=True)
        batch_results.append(result)

    # Validation hier bewusst mit evaluate auf val_ds:
    # val_ds ist ebenfalls gebatcht, aber nicht geshuffelt.
    val_result = simple_loop_model.evaluate(val_ds, verbose=0, return_dict=True)

    row = {
        "epoch": epoch,
        "train_loss_last_batch": batch_results[-1]["loss"],
        "train_accuracy_last_batch": batch_results[-1]["accuracy"],
        "val_loss": val_result["loss"],
        "val_accuracy": val_result["accuracy"],
    }
    simple_history.append(row)

pd.DataFrame(simple_history)


Diese Zwischenstufe ist oft praktisch, wenn man nur etwas mehr Kontrolle als `model.fit(...)` möchte. Für viele Projekte reicht aber `model.fit(...)` plus Callbacks.

Der nächste Abschnitt geht tiefer: Dort schreiben wir mit `GradientTape` auch den Trainingsschritt selbst.


## Ein Trainingsschritt

`GradientTape` beobachtet die Rechenoperationen im `with`-Block. Danach kann TensorFlow die Gradienten des Losses bezüglich der trainierbaren Gewichte berechnen.

Der Ablauf im Detail:

1. `model(x_batch, training=True)` berechnet Vorhersagen für den Batch.
2. `loss_fn(y_batch, y_pred)` berechnet den Fehler für diesen Batch.
3. `tape.gradient(loss, model.trainable_variables)` berechnet für jedes trainierbare Gewicht den passenden Gradienten.
4. `zip(gradients, model.trainable_variables)` paart jeden Gradienten mit genau der Variable, zu der er gehört.
5. `optimizer.apply_gradients(...)` verändert die Gewichte. Hier wirken Learning Rate, Adam, SGD, Momentum usw.

`zip` ist normales Python: Aus `[g1, g2]` und `[w1, w2]` wird `[(g1, w1), (g2, w2)]`. Keras erwartet genau solche Paare: welcher Gradient soll auf welches Gewicht angewendet werden?


In [ ]:
@tf.function
def train_step(x_batch, y_batch):
    # 1. TensorFlow merkt sich alle Operationen im Tape-Block,
    #    die vom Loss zurück zu den Modellgewichten führen.
    with tf.GradientTape() as tape:
        # training=True ist wichtig für Layer wie Dropout oder BatchNorm.
        y_pred = model(x_batch, training=True)
        loss = loss_fn(y_batch, y_pred)

    # 2. Für jede trainierbare Variable wird ein Gradient berechnet.
    #    Die Reihenfolge passt zu model.trainable_variables.
    gradients = tape.gradient(loss, model.trainable_variables)

    # 3. zip baut Paare: (Gradient, Variable).
    #    apply_gradients übergibt diese Paare an den Optimizer.
    #    Der Optimizer entscheidet dann mit seiner Learning Rate und seinem Verfahren
    #    (z. B. Adam, SGD, Momentum), wie stark jede Variable geändert wird.
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))

    # 4. Metriken werden nur aktualisiert. Sie ändern keine Modellgewichte.
    train_loss.update_state(loss)
    train_acc.update_state(y_batch, y_pred)


@tf.function
def val_step(x_batch, y_batch):
    # In der Validation werden keine Gradienten berechnet und keine Gewichte geändert.
    y_pred = model(x_batch, training=False)
    loss = loss_fn(y_batch, y_pred)
    val_loss.update_state(loss)
    val_acc.update_state(y_batch, y_pred)


## Was genau bekommt der Optimizer?

Die folgende Zelle zeigt einmal explizit, welche Variablen und Gradienten gepaart werden. Das ist genau das, was `zip(gradients, model.trainable_variables)` im Trainingsschritt erzeugt.


In [ ]:
# Wir nehmen genau einen Batch aus train_ds, nur um die Zuordnung zu zeigen.
# train_ds selbst bleibt danach weiter nutzbar; das ist keine komplette Epoche.
example_x, example_y = next(iter(train_ds))
with tf.GradientTape() as tape:
    example_pred = model(example_x, training=True)
    example_loss = loss_fn(example_y, example_pred)

# Für diesen einen Batch berechnen wir die Gradienten.
example_gradients = tape.gradient(example_loss, model.trainable_variables)

# Diese Tabelle zeigt: Jede trainierbare Variable hat genau einen passenden Gradienten.
pd.DataFrame(
    {
        "variable": [variable.name for variable in model.trainable_variables],
        "variable_shape": [str(variable.shape) for variable in model.trainable_variables],
        "gradient_shape": [str(gradient.shape) for gradient in example_gradients],
    }
)


## Komplette Trainingsschleife

Wir resetten die Metriken zu Beginn jeder Epoche. Danach laufen wir über alle Trainingsbatches und anschließend über alle Validation-Batches.


In [ ]:
history = []
epochs = 50

for epoch in range(1, epochs + 1):
    # Metriken sammeln Werte über viele Batches.
    # reset_state() startet die Zählung für die neue Epoche wieder bei 0.
    train_loss.reset_state()
    train_acc.reset_state()
    val_loss.reset_state()
    val_acc.reset_state()

    # Training: Jeder Batch wird an train_step übergeben.
    # In train_step werden Gradienten berechnet und Gewichte verändert.
    for x_batch, y_batch in train_ds:
        train_step(x_batch, y_batch)

    # Validation: Jeder Batch wird an val_step übergeben.
    # Hier werden keine Gewichte verändert, nur Metriken berechnet.
    for x_batch, y_batch in val_ds:
        val_step(x_batch, y_batch)

    row = {
        "epoch": epoch,
        "loss": float(train_loss.result().numpy()),
        "accuracy": float(train_acc.result().numpy()),
        "val_loss": float(val_loss.result().numpy()),
        "val_accuracy": float(val_acc.result().numpy()),
    }
    history.append(row)

    if epoch == 1 or epoch % 10 == 0:
        print(
            f"Epoche {epoch:02d}: loss={row['loss']:.3f}, acc={row['accuracy']:.3f}, "
            f"val_loss={row['val_loss']:.3f}, val_acc={row['val_accuracy']:.3f}"
        )

history_df = pd.DataFrame(history)
history_df.tail()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
history_df.plot(x="epoch", y=["loss", "val_loss"], ax=axes[0])
history_df.plot(x="epoch", y=["accuracy", "val_accuracy"], ax=axes[1])
axes[0].set_title("Loss")
axes[1].set_title("Accuracy")


## Blick in die Gradienten

Gradienten sind Tensoren mit derselben Form wie die jeweiligen Gewichte. Große oder `nan`-Gradienten sind ein Warnsignal für instabiles Training.


In [ ]:
# Wieder nur ein einzelner Kontroll-Batch.
# Hier schauen wir nicht auf die Shapes, sondern auf die Größe der Gradienten.
x_batch, y_batch = next(iter(train_ds))
with tf.GradientTape() as tape:
    y_pred = model(x_batch, training=True)
    loss = loss_fn(y_batch, y_pred)

gradients = tape.gradient(loss, model.trainable_variables)

pd.DataFrame(
    {
        "variable": [v.name for v in model.trainable_variables],
        "shape": [str(v.shape) for v in model.trainable_variables],
        "gradient_norm": [float(tf.linalg.global_norm([g]).numpy()) for g in gradients],
    }
)


## Wann braucht man eigene Trainingsschleifen?

Typische Gründe:

- mehrere Optimizer,
- mehrere Losses mit eigener Logik,
- GANs, Reinforcement Learning oder Self-Supervised Learning,
- eigene Logging- oder Sampling-Schritte pro Batch,
- Forschungscode, der nicht gut in `model.fit(...)` passt.

Für Standardklassifikation und Standardregression ist `model.fit(...)` meist einfacher, stabiler und besser wartbar.


## Limitierungen

- Eine eigene Trainingsschleife gibt viel Kontrolle, aber auch viel Verantwortung. Callbacks, Checkpoints, Early Stopping und History-Objekt kommen nicht automatisch.
- Metriken müssen korrekt zurückgesetzt werden. Sonst vermischen sich Epochen.
- `@tf.function` macht den Loop schneller, kann aber Fehlermeldungen schwieriger lesbar machen. Beim Debuggen kann man es vorübergehend entfernen.
- Nicht jeder Python-Ausdruck funktioniert sauber in einem TensorFlow-Graph. TensorFlow-Operationen sind robuster.
- Der Loop hier ist didaktisch einfach. Produktiv würde man Checkpointing, Logging, Seed-Kontrolle und Fehlerbehandlung ergänzen.
